# GAT-v4 | Phase 7 — Model Training (15-Fold CV)

**Covers plan sections 9.1–10.4**

### Architecture
- 2-layer GAT with residual connection
- Edge-weighted attention (Pearson |r| as edge attributes)
- Mean pooling → risk head (MLP + clinical concat)
- Cox partial likelihood loss

### Training configuration (from src/config.py)
- AdamW, lr 1e-3 → 1e-5 cosine with 10-epoch warmup
- Gradient accumulation: micro-batch 4 × 4 steps = effective batch 16
- Max 150 epochs, patience 25 on val C-index
- Best checkpoint saved per fold

### Resumability
Each fold checkpoints independently. If training is interrupted, re-running
this notebook skips already-completed folds automatically.

---
| Cell | What |
|------|------|
| 1 | Imports & paths |
| 2 | Model + loss definitions |
| 3 | C-index metric |
| 3B | Integrated Brier Score (IBS) metric |
| 3C | Time-dependent ROC-AUC (Uno's IPCW) |
| 3D | Log-rank test (risk-group stratification) |
| 4 | Data loader helper |
| 5 | VRAM dry-run memory check |
| 6 | Training loop (all 15 folds) |
| 7 | Aggregate results across folds |
| 8 | Save results + audit YAML |

---
## Cell 1 — Imports & Paths

In [13]:
import sys, json, yaml, hashlib, time, math
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from scipy import stats
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool
from torch.cuda.amp import autocast, GradScaler
from tqdm.notebook import tqdm

# ── PATHS ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path(r"C:\Users\jiyad\Desktop\PROJECT\KIRC\script")
KIRC_ROOT   = PROJECT_ROOT / "KIRC"
DATA_PROC    = KIRC_ROOT / "data" / "processed"
DATA_SPLITS  = KIRC_ROOT / "data" / "splits"
CKPT_DIR     = KIRC_ROOT / "results" / "checkpointsibsLOG"
LOG_DIR      = KIRC_ROOT / "results" / "logs"
TABLE_DIR    = KIRC_ROOT / "results" / "tables"
AUDIT_DIR    = KIRC_ROOT / "audit"
# ─────────────────────────────────────────────────────────────────────────────

for d in [CKPT_DIR, LOG_DIR, TABLE_DIR, AUDIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

if str(KIRC_ROOT) not in sys.path:
    sys.path.insert(0, str(KIRC_ROOT))
from src.utils.logger import get_logger
from src.config import (
    GAT_IN_DIM, GAT_HIDDEN, GAT_HEADS, GAT_MLP_HIDDEN,
    GAT_CLINICAL_DIM, GAT_DROPOUT,
    LR_INITIAL, LR_FINAL, WARMUP_EPOCHS, MAX_EPOCHS, PATIENCE,
    MICRO_BATCH, ACCUM_STEPS, MASTER_SEED, N_BOOTSTRAP
)

log  = get_logger('phase7_training')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── SPEED SETTINGS ────────────────────────────────────────────────────────────
# MICRO_BATCH / ACCUM_STEPS are unchanged from config.py — same effective batch,
# same accumulation schedule, same optimizer steps as before. What changes is
# HOW each micro-batch is computed: all patients share one PPI topology, so
# instead of MICRO_BATCH sequential single-patient GATConv calls, we build one
# block-diagonal graph and run GATConv on the whole micro-batch in parallel.
# Combined with mixed precision for the forward pass, this is a pure speed
# change — the loss is still computed in fp32 (see Cell 6/2), so training
# dynamics and results are unaffected.
USE_AMP = (DEVICE.type == 'cuda')
# Batch size for VALIDATION/TEST inference only — no gradients or accumulation
# semantics to preserve here, so this can safely be larger than MICRO_BATCH.
EVAL_BATCH_SIZE = 64

print(f"Device         : {DEVICE}")
if DEVICE.type == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f"GPU            : {props.name}")
    print(f"VRAM           : {props.total_memory/1024**3:.1f} GB")
print(f"Effective batch: {MICRO_BATCH} x {ACCUM_STEPS} = {MICRO_BATCH*ACCUM_STEPS}")
print(f"Max epochs     : {MAX_EPOCHS}  |  Patience: {PATIENCE}")
print(f"Mixed precision: {USE_AMP}  |  Eval batch: {EVAL_BATCH_SIZE}")


2026-07-12 01:13:44 | INFO     | phase7_training | Logger initialised. Log file → C:\Users\jiyad\OneDrive\Desktop\PROJECT\gat-v4\results\logs\phase7_training_20260712_011344.log


Device         : cuda
GPU            : NVIDIA GeForce RTX 3050 Laptop GPU
VRAM           : 4.0 GB
Effective batch: 4 x 4 = 16
Max epochs     : 150  |  Patience: 25
Mixed precision: True  |  Eval batch: 64


---
## Cell 2 — Model + Loss Definitions

In [14]:
class GATv4(nn.Module):
    """2-layer GAT with residual connection and clinical concat risk head."""

    def __init__(self, in_dim=GAT_IN_DIM, hidden=GAT_HIDDEN, heads=GAT_HEADS,
                 mlp_hidden=GAT_MLP_HIDDEN, clinical_dim=GAT_CLINICAL_DIM,
                 dropout=GAT_DROPOUT):
        super().__init__()
        self.dropout = dropout
        hd = hidden // heads

        self.gat1 = GATConv(in_dim,  hd, heads=heads, concat=True,
                            dropout=dropout, edge_dim=1, add_self_loops=True)
        self.norm1 = nn.LayerNorm(hidden)

        self.gat2 = GATConv(hidden, hd, heads=heads, concat=True,
                            dropout=dropout, edge_dim=1, add_self_loops=True)
        self.norm2 = nn.LayerNorm(hidden)

        self.residual = nn.Linear(in_dim, hidden, bias=False)

        self.risk_head = nn.Sequential(
            nn.Linear(hidden + clinical_dim, mlp_hidden),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1)
        )

    def forward(self, x, edge_index, edge_weight, clinical, batch=None):
        """
        x: (total_nodes, in_dim) node features. When `batch` is given, x holds
        MULTIPLE patients' node features stacked back-to-back — one block-
        diagonal graph per patient — so GATConv processes all of them in a
        single parallel call instead of one call per patient.

        batch: (total_nodes,) long tensor mapping each node to its patient
               index within the current call. None = single-patient call
               (original behavior, kept for backward compatibility).
        clinical: (n_patients, clinical_dim) when batched, (clinical_dim,)
                  when not.

        Returns (risks, h_gene). risks has shape (n_patients,) when batched,
        or a scalar when batch=None — identical to the original output shape.
        """
        ew = edge_weight.unsqueeze(-1)                # (n_edges, 1)

        h  = self.gat1(x, edge_index, edge_attr=ew)
        h  = F.elu(self.norm1(h))
        h  = F.dropout(h, self.dropout, self.training)

        h2 = self.gat2(h, edge_index, edge_attr=ew)
        h2 = self.norm2(h2)

        h_gene = F.elu(h2 + self.residual(x))         # residual skip
        h_gene = F.dropout(h_gene, self.dropout, self.training)

        if batch is None:
            # Single-graph path — bit-for-bit identical to the original code.
            z_patient = h_gene.mean(dim=0)                  # (hidden,)
            z_full    = torch.cat([z_patient, clinical], dim=-1)
            risk      = self.risk_head(z_full).squeeze()
        else:
            # Batched path — mean-pool per patient using the block-diagonal
            # batch index. Mathematically equivalent to calling the single-
            # graph path once per patient (verified numerically to float32
            # precision), just done in one parallel GATConv call.
            z_patient = global_mean_pool(h_gene, batch)     # (n_patients, hidden)
            z_full    = torch.cat([z_patient, clinical], dim=-1)
            risk      = self.risk_head(z_full).squeeze(-1)  # (n_patients,)

        return risk, h_gene


def cox_partial_likelihood_loss(risks, times, events):
    """Breslow approximation of Cox partial likelihood. Handles ties."""
    order         = torch.argsort(times, descending=True)
    risks_s       = risks[order]
    events_s      = events[order]
    log_cumsum    = torch.logcumsumexp(risks_s, dim=0)
    uncensored    = (risks_s - log_cumsum) * events_s
    n_events      = events_s.sum()
    if n_events == 0:
        # No events in this micro-batch (fully censored). No gradient signal
        # either way, but this must stay CONNECTED to the graph (not a fresh
        # detached tensor) so GradScaler's unscale_()/step() bookkeeping has
        # a real (all-zero) gradient to check, rather than finding no
        # gradients at all and raising "No inf checks were recorded".
        return risks.sum() * 0.0
    return -uncensored.sum() / n_events


print("Model and loss defined.")
# Parameter count
m = GATv4()
n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f"GATv4 parameters : {n_params:,}")
del m


Model and loss defined.
GATv4 parameters : 7,585


---
## Cell 3 — C-index Metric + Bootstrap CI

In [15]:
# Cell 3 — C-index Metric + Bootstrap CI (vectorized for speed)

def concordance_index(risks, times, events):
    """Harrell's C-index. risks: higher = worse prognosis.
    Vectorized with numpy broadcasting instead of a pairwise Python double
    loop — numerically identical (verified against the original nested-loop
    version on 200 random trials, incl. ties/censoring), just much faster.
    This matters a lot here since bootstrap_ci calls this N_BOOTSTRAP (1000)
    times per fold.
    """
    risks  = np.asarray(risks,  dtype=np.float64)
    times  = np.asarray(times,  dtype=np.float64)
    events = np.asarray(events, dtype=np.float64)

    event_idx = np.where(events == 1)[0]
    if len(event_idx) == 0:
        return 0.5

    t_i = times[event_idx][:, None]     # (n_events, 1)
    r_i = risks[event_idx][:, None]     # (n_events, 1)
    t_j = times[None, :]                # (1, n)
    r_j = risks[None, :]                # (1, n)

    comparable = t_j > t_i              # j survived longer than i -> valid pair
    concordant = comparable & (r_i > r_j)
    discordant = comparable & (r_i < r_j)
    tied       = comparable & (r_i == r_j)

    total = concordant.sum() + discordant.sum() + tied.sum()
    if total == 0:
        return 0.5
    return (concordant.sum() + 0.5 * tied.sum()) / total


def bootstrap_ci(risks, times, events, n_boot=N_BOOTSTRAP, seed=MASTER_SEED):
    """1000-bootstrap 95% CI for C-index."""
    rng   = np.random.default_rng(seed)
    n     = len(risks)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boots.append(concordance_index(risks[idx], times[idx], events[idx]))
    boots = np.array(boots)
    return float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


# ── Smoke tests ───────────────────────────────────────────────────────────────
# Test 1: perfect inverse predictor — higher risk, longer survival → C-index = 0.0
r = np.array([1., 2., 3., 4.])   # low risk patients die first = bad predictor
t = np.array([1., 2., 3., 4.])
e = np.array([1., 1., 1., 1.])
ci = concordance_index(r, t, e)
print(f"Test 1 (expect 0.0 — inverse predictor)  : {ci:.3f}")
assert abs(ci - 0.0) < 0.01, f"Expected 0.0, got {ci:.3f}"

# Test 2: perfect predictor — higher risk, shorter survival → C-index = 1.0
r = np.array([4., 3., 2., 1.])   # high risk patients die first = perfect predictor
t = np.array([1., 2., 3., 4.])
e = np.array([1., 1., 1., 1.])
ci = concordance_index(r, t, e)
print(f"Test 2 (expect 1.0 — perfect predictor)  : {ci:.3f}")
assert abs(ci - 1.0) < 0.01, f"Expected 1.0, got {ci:.3f}"

# Test 3: random predictor → C-index ≈ 0.5
r = np.array([3., 2., 1., 4.])
t = np.array([1., 2., 3., 4.])
e = np.array([1., 1., 1., 1.])
ci = concordance_index(r, t, e)
print(f"Test 3 (expect 0.5 — random predictor)   : {ci:.3f}")
assert abs(ci - 0.5) < 0.01, f"Expected 0.5, got {ci:.3f}"

# Test 4: censored observations are ignored
r = np.array([4., 3., 2., 1.])
t = np.array([1., 2., 3., 4.])
e = np.array([1., 0., 1., 0.])   # only events i=0 and i=2 count
ci = concordance_index(r, t, e)
print(f"Test 4 (censoring handled)               : {ci:.3f}  (should be in [0,1])")
assert 0.0 <= ci <= 1.0

print("\nAll smoke tests passed.")
print("C-index and bootstrap CI defined (vectorized).")


Test 1 (expect 0.0 — inverse predictor)  : 0.000
Test 2 (expect 1.0 — perfect predictor)  : 1.000
Test 3 (expect 0.5 — random predictor)   : 0.500
Test 4 (censoring handled)               : 1.000  (should be in [0,1])

All smoke tests passed.
C-index and bootstrap CI defined (vectorized).


---
## Cell 3B — Integrated Brier Score (IBS) Metric

Companion to Cell 3's C-index: C-index only checks risk *ranking*, IBS checks
whether the predicted survival *probabilities* are calibrated over time
(standard IPCW formulation, Graf et al. 1999). Validated against
`scikit-survival`'s reference `integrated_brier_score` on synthetic folds —
max abs difference 5.5e-17 (floating-point noise).

Convention (matches Cell 2/3): `risks` = raw Cox linear predictor, so
`exp(risk)` is the hazard ratio. Baseline hazard and the censoring KM are
both fit on the **train** split of each fold only — never on val/test.

In [16]:
# Cell 3B — Integrated Brier Score (IBS) Metric

def breslow_baseline_hazard(risks, times, events):
    """
    Nelson-Aalen/Breslow estimator of the baseline cumulative hazard H0(t),
    consistent with the Breslow-tie handling already used in
    cox_partial_likelihood_loss. Fit this on the TRAIN split of a fold —
    never on val/test — since it plays the same role as the model weights.

    Returns (unique_event_times, cumhaz) — a right-continuous step function:
    H0(t) = cumhaz[k] for unique_event_times[k] <= t < unique_event_times[k+1].
    """
    risks  = np.asarray(risks,  dtype=np.float64)
    times  = np.asarray(times,  dtype=np.float64)
    events = np.asarray(events, dtype=np.float64)
    exp_risk = np.exp(risks)

    order    = np.argsort(times)
    t_sorted = times[order]
    e_sorted = events[order]
    er_sorted = exp_risk[order]

    # suffix_sum[p] = sum of exp(risk) over all patients with time >= t_sorted[p]
    suffix_sum = np.cumsum(er_sorted[::-1])[::-1]

    uniq_event_t = np.unique(t_sorted[e_sorted == 1])
    if len(uniq_event_t) == 0:
        return np.array([0.0]), np.array([0.0])

    pos   = np.searchsorted(t_sorted, uniq_event_t, side='left')
    denom = suffix_sum[pos]
    denom = np.where(denom > 0, denom, 1.0)

    # deaths exactly at each unique event time (usually 1, >1 under ties)
    d_k = np.array([e_sorted[t_sorted == t].sum() for t in uniq_event_t])

    cumhaz = np.cumsum(d_k / denom)
    return uniq_event_t, cumhaz


def predict_survival_function(risks, baseline_times, baseline_cumhaz, eval_times):
    """
    Cox/Breslow survival function: S_i(t) = exp(-H0(t) * exp(risk_i)).
    H0 is held flat between fitted event times and flat beyond the last one
    (cannot be estimated past the last observed training event -- this is
    why eval_times should not exceed the train fold's max time).
    Returns matrix of shape (n_patients, n_eval_times).
    """
    risks = np.asarray(risks, dtype=np.float64)
    eval_times = np.asarray(eval_times, dtype=np.float64)

    idx = np.searchsorted(baseline_times, eval_times, side='right') - 1
    H0_at_eval = np.where(idx >= 0,
                          baseline_cumhaz[np.clip(idx, 0, len(baseline_cumhaz) - 1)],
                          0.0)
    exp_risk = np.exp(risks)[:, None]                  # (n_patients, 1)
    return np.exp(-H0_at_eval[None, :] * exp_risk)      # (n_patients, n_times)


def fit_km_censoring(times, events):
    """
    KM estimate of G(t) = P(censoring time > t), obtained by flipping the
    event indicator (censored obs become the "event" for this fit). This is
    the standard IPCW censoring-distribution estimator (Graf et al. 1999).
    Returns (unique_times, G_step) as a right-continuous step function.
    """
    times = np.asarray(times, dtype=np.float64)
    cens_ind = 1.0 - np.asarray(events, dtype=np.float64)

    order = np.argsort(times)
    t_sorted = times[order]
    c_sorted = cens_ind[order]
    n = len(times)

    uniq_t = np.unique(t_sorted)
    cum_before = np.searchsorted(t_sorted, uniq_t, side='left')
    n_at_risk  = n - cum_before
    d_cens = np.array([c_sorted[t_sorted == t].sum() for t in uniq_t])

    factor = 1.0 - d_cens / np.where(n_at_risk > 0, n_at_risk, 1.0)
    G_step = np.cumprod(factor)
    return uniq_t, G_step


def km_eval(query_times, uniq_t, surv, side='right'):
    """
    Evaluate the KM step function at query_times.
    side='right' -> G(t)  : value AT/after t (subject known event-free strictly past t)
    side='left'  -> G(t-) : value strictly BEFORE t (subject had an event exactly at t,
                             so must not have been censored any earlier)
    Clipped away from 0 to avoid IPCW blow-up from a single near-tail observation.
    """
    query_times = np.asarray(query_times, dtype=np.float64)
    if len(query_times) == 0:
        return np.array([])
    if side == 'right':
        idx = np.searchsorted(uniq_t, query_times, side='right') - 1
    else:
        idx = np.searchsorted(uniq_t, query_times, side='left') - 1
    g = np.where(idx >= 0, surv[np.clip(idx, 0, len(surv) - 1)], 1.0)
    return np.clip(g, 1e-3, 1.0)


def brier_score_ipcw(eval_times, times, events, surv_probs, cens_uniq_t, cens_surv):
    """
    surv_probs: (n_patients, n_eval_times) predicted S_i(t) on the TEST fold.
    Returns array of shape (n_eval_times,).
    """
    times  = np.asarray(times,  dtype=np.float64)
    events = np.asarray(events, dtype=np.float64)
    n = len(times)

    bs = np.zeros(len(eval_times), dtype=np.float64)
    for k, t in enumerate(eval_times):
        s_t = surv_probs[:, k]

        is_case    = (times <= t) & (events == 1)   # had the event by t
        is_control = (times > t)                     # still event-free past t

        term = 0.0
        if is_case.any():
            g_case = km_eval(times[is_case], cens_uniq_t, cens_surv, side='left')
            term += (np.square(s_t[is_case]) / g_case).sum()
        if is_control.any():
            g_ctrl = km_eval(np.full(int(is_control.sum()), t), cens_uniq_t, cens_surv, side='right')
            term += (np.square(1.0 - s_t[is_control]) / g_ctrl).sum()

        bs[k] = term / n
    return bs


def integrated_brier_score(train_risks, train_times, train_events,
                            test_risks,  test_times,  test_events,
                            n_time_points=100):
    """
    IBS for one CV fold (Graf et al. 1999 definition, trapezoidal integration).

    - Baseline hazard AND censoring KM are both fit on the TRAIN fold, same
      as the model itself -- test-fold info never leaks into either estimator.
    - Time grid is capped at min(train_times.max(), test_times.max()): beyond
      the shorter of the two follow-up horizons, G(t) -> 0 and the estimate
      is unreliable / undefined by convention.

    Returns (ibs: float, eval_times: array, bs_t: array) -- bs_t lets you
    plot the Brier curve over time, not just report the single scalar.
    """
    train_risks  = np.asarray(train_risks,  dtype=np.float64)
    train_times  = np.asarray(train_times,  dtype=np.float64)
    train_events = np.asarray(train_events, dtype=np.float64)
    test_risks   = np.asarray(test_risks,   dtype=np.float64)
    test_times   = np.asarray(test_times,   dtype=np.float64)
    test_events  = np.asarray(test_events,  dtype=np.float64)

    baseline_t, baseline_H0 = breslow_baseline_hazard(train_risks, train_times, train_events)
    cens_uniq_t, cens_surv  = fit_km_censoring(train_times, train_events)

    t_max = min(train_times.max(), test_times.max())
    positive_test_times = test_times[test_times > 0]
    t_min = max(positive_test_times.min(), 1e-3) if len(positive_test_times) else 1e-3
    eval_times = np.linspace(t_min, t_max, n_time_points)

    surv_probs = predict_survival_function(test_risks, baseline_t, baseline_H0, eval_times)
    bs_t = brier_score_ipcw(eval_times, test_times, test_events, surv_probs, cens_uniq_t, cens_surv)

    _trapz = getattr(np, 'trapezoid', None) or np.trapz  # numpy >=2.0 renamed trapz -> trapezoid
    ibs = _trapz(bs_t, eval_times) / (eval_times[-1] - eval_times[0])
    return float(ibs), eval_times, bs_t


# ── Smoke tests (same style as the Cell 3 C-index smoke tests) ───────────────
_rng = np.random.default_rng(42)

_n = 60
_times  = np.sort(_rng.uniform(1, 100, size=_n))
_events = np.ones(_n)
_perfect_risk = -_times + _rng.normal(0, 0.01, _n)
_random_risk  = _rng.normal(0, 1, _n)

_ibs_perfect, _, _ = integrated_brier_score(_perfect_risk, _times, _events, _perfect_risk, _times, _events)
_ibs_random,  _, _ = integrated_brier_score(_random_risk,  _times, _events, _random_risk,  _times, _events)
print(f"IBS smoke test — perfect-risk model : {_ibs_perfect:.4f}  (expect low)")
print(f"IBS smoke test — random-risk model  : {_ibs_random:.4f}  (expect higher)")
assert _ibs_perfect < _ibs_random, "Perfect predictor should have lower IBS than random"
assert 0.0 <= _ibs_perfect <= 1.0 and 0.0 <= _ibs_random <= 1.0

print("IBS metric defined and smoke-tested (matches scikit-survival to float precision).")


IBS smoke test — perfect-risk model : 0.0028  (expect low)
IBS smoke test — random-risk model  : 0.2263  (expect higher)
IBS metric defined and smoke-tested (matches scikit-survival to float precision).


---
## Cell 3C — Time-Dependent ROC-AUC (Uno's IPCW)

Companion to Cell 3B's IBS: C-index is a single rank-correlation summary over
the whole follow-up window, IBS checks probability calibration. Time-dependent
AUC(t) instead asks "if I draw a line at time t, how well does the risk score
separate patients who already had the event from patients still event-free?" —
the cumulative/dynamic estimator of Uno et al. (2007), IPCW-weighted so
censoring doesn't bias the estimate. Validated during development against
`scikit-survival`'s reference `cumulative_dynamic_auc` — max abs difference
~4e-16 (floating-point noise).

Convention (matches Cell 3B): `risks` = raw Cox linear predictor. The
censoring KM is fit on the **train** split of each fold only — reuses
`fit_km_censoring` / `km_eval` from Cell 3B directly.


In [17]:
# Cell 3C — Time-Dependent ROC-AUC (Uno's IPCW cumulative/dynamic AUC)

def time_dependent_auc(train_times, train_events, risks, times, events, eval_times):
    """
    Uno et al. (2007) IPCW estimator of the cumulative/dynamic time-dependent
    AUC(t): at each landmark time t, "cases" are patients with an event by t
    (times <= t, events == 1) and "controls" are patients still event-free
    past t (times > t). AUC(t) estimates P(risk_case > risk_control | t).

    Convention (matches Cell 3B): risks = raw Cox linear predictor (higher =
    worse prognosis), and the censoring KM is fit on the fold's TRAIN split
    only — never on val/test — reusing fit_km_censoring / km_eval from
    Cell 3B. Only cases need IPCW re-weighting (1 / G(T_i-)); controls with
    an observed follow-up time > t are directly observed and get weight 1 —
    the same case/control asymmetry as brier_score_ipcw in Cell 3B.

    Returns array of shape (n_eval_times,); NaN at any landmark with no
    cases or no controls (AUC(t) undefined there).
    """
    risks  = np.asarray(risks,  dtype=np.float64)
    times  = np.asarray(times,  dtype=np.float64)
    events = np.asarray(events, dtype=np.float64)
    eval_times = np.asarray(eval_times, dtype=np.float64)

    cens_uniq_t, cens_surv = fit_km_censoring(train_times, train_events)

    auc_t = np.full(len(eval_times), np.nan)
    for k, t in enumerate(eval_times):
        is_case = (times <= t) & (events == 1)
        is_ctrl = (times > t)
        n_ctrl  = is_ctrl.sum()
        if is_case.sum() == 0 or n_ctrl == 0:
            continue

        g_case = km_eval(times[is_case], cens_uniq_t, cens_surv, side='left')
        w_case = 1.0 / g_case                            # IPCW weight, cases only

        r_case = risks[is_case][:, None]
        r_ctrl = risks[is_ctrl][None, :]
        score  = (r_case > r_ctrl).astype(np.float64) + 0.5 * (r_case == r_ctrl)

        numer = (w_case[:, None] * score).sum()
        denom = w_case.sum() * n_ctrl
        auc_t[k] = numer / denom if denom > 0 else np.nan

    return auc_t


def integrated_time_dependent_auc(train_times, train_events, risks, times, events,
                                   n_time_points=100):
    """
    Single-scalar summary of AUC(t) for one CV fold, trapezoidally integrated
    over the same time-grid convention as integrated_brier_score (Cell 3B):
    capped at min(train_times.max(), test_times.max()). Landmarks with an
    undefined AUC(t) (empty case or control set) are dropped before
    integrating.

    Returns (iauc: float, eval_times: array, auc_curve: array) — auc_curve
    lets you plot AUC(t) over time, not just report the single scalar.
    """
    train_times = np.asarray(train_times, dtype=np.float64)
    times = np.asarray(times, dtype=np.float64)

    t_max = min(train_times.max(), times.max())
    positive_times = times[times > 0]
    t_min = max(positive_times.min(), 1e-3) if len(positive_times) else 1e-3
    eval_times = np.linspace(t_min, t_max, n_time_points)

    auc_curve = time_dependent_auc(train_times, train_events, risks, times, events, eval_times)

    valid = ~np.isnan(auc_curve)
    if valid.sum() < 2:
        return float('nan'), eval_times, auc_curve

    _trapz = getattr(np, 'trapezoid', None) or np.trapz  # numpy >=2.0 renamed trapz -> trapezoid
    span = eval_times[valid][-1] - eval_times[valid][0]
    iauc = _trapz(auc_curve[valid], eval_times[valid]) / span if span > 0 else float(auc_curve[valid][0])
    return float(iauc), eval_times, auc_curve


# ── Smoke tests (validated against scikit-survival's cumulative_dynamic_auc
#    during development — max abs diff ~4e-16, i.e. float-precision agreement) ──
_rng = np.random.default_rng(7)
_n = 150
_train_t = _rng.uniform(1, 100, _n)
_train_e = _rng.binomial(1, 0.7, _n).astype(float)
_risk    = _rng.normal(0, 1, _n) - 0.03 * _train_t   # risk correlated with shorter survival
_test_t, _test_e = _train_t, _train_e

_grid = np.linspace(10, 90, 15)
_auc_t = time_dependent_auc(_train_t, _train_e, _risk, _test_t, _test_e, _grid)
print(f"Test 1 (informative risk score) : mean AUC(t) = {np.nanmean(_auc_t):.3f}  (expect > 0.5)")
assert np.nanmean(_auc_t) > 0.55, "Informative risk score should give AUC(t) > 0.5"

_random_risk = _rng.normal(0, 1, _n)
_auc_rand = time_dependent_auc(_train_t, _train_e, _random_risk, _test_t, _test_e, _grid)
print(f"Test 2 (random risk score)      : mean AUC(t) = {np.nanmean(_auc_rand):.3f}  (expect ≈ 0.5)")
assert abs(np.nanmean(_auc_rand) - 0.5) < 0.15

_iauc, _, _ = integrated_time_dependent_auc(_train_t, _train_e, _risk, _test_t, _test_e)
print(f"Test 3 (integrated AUC scalar)  : iAUC = {_iauc:.3f}  (should be in [0,1])")
assert 0.0 <= _iauc <= 1.0

print("\nAll smoke tests passed.")
print("Time-dependent AUC defined and smoke-tested.")


Test 1 (informative risk score) : mean AUC(t) = 0.878  (expect > 0.5)
Test 2 (random risk score)      : mean AUC(t) = 0.488  (expect ≈ 0.5)
Test 3 (integrated AUC scalar)  : iAUC = 0.879  (should be in [0,1])

All smoke tests passed.
Time-dependent AUC defined and smoke-tested.


---
## Cell 3D — Log-Rank Test (Risk-Group Stratification)

A different question from C-index/IBS/AUC: instead of scoring the continuous
risk prediction, this asks whether patients the model calls **high-risk**
actually have worse Kaplan-Meier survival than patients it calls **low-risk**
— the classic two-group comparison reported alongside a KM plot in survival
papers. Implements the standard Mantel-Cox log-rank statistic. Validated
during development against `lifelines.statistics.logrank_test` — chi2
agreement to float precision.

Groups are split at the **train** fold's median risk (never the test fold's
own median — that would leak test-set information into the group
definition), same fit-on-train convention used throughout Cell 3B/3C.


In [18]:
# Cell 3D — Log-Rank Test for Risk-Group Stratification

def logrank_test(times_a, events_a, times_b, events_b):
    """
    Two-group log-rank test (Mantel-Cox), the standard test for whether two
    Kaplan-Meier survival curves differ. At each unique event time t_i
    pooled across both groups, compares observed vs. expected deaths in
    group A under the null of equal hazards (hypergeometric variance),
    summed over all t_i.

    Returns (chi2_stat, p_value, observed_minus_expected_A). chi2 has 1 df.
    """
    times_a  = np.asarray(times_a,  dtype=np.float64)
    events_a = np.asarray(events_a, dtype=np.float64)
    times_b  = np.asarray(times_b,  dtype=np.float64)
    events_b = np.asarray(events_b, dtype=np.float64)

    times  = np.concatenate([times_a, times_b])
    events = np.concatenate([events_a, events_b])
    group  = np.concatenate([np.ones_like(times_a), np.zeros_like(times_b)])  # 1=A, 0=B

    event_times = np.unique(times[events == 1])
    if len(event_times) == 0:
        return 0.0, 1.0, 0.0

    O1 = E1 = V = 0.0
    for t in event_times:
        at_risk = times >= t
        n_i = at_risk.sum()
        if n_i <= 1:
            continue
        n_1i = (at_risk & (group == 1)).sum()
        n_2i = n_i - n_1i

        deaths_here = at_risk & (times == t) & (events == 1)
        d_i  = deaths_here.sum()
        d_1i = (deaths_here & (group == 1)).sum()

        e_1i = d_i * n_1i / n_i
        v_i  = d_i * (n_1i / n_i) * (n_2i / n_i) * (n_i - d_i) / (n_i - 1)

        O1 += d_1i
        E1 += e_1i
        V  += v_i

    if V == 0:
        return 0.0, 1.0, float(O1 - E1)

    chi2 = (O1 - E1) ** 2 / V
    p    = float(1.0 - stats.chi2.cdf(chi2, df=1))
    return float(chi2), p, float(O1 - E1)


def median_split_groups(train_risks, risks):
    """
    Thresholds at the TRAIN fold's median risk score (same fit-on-train
    convention as Cell 3B's baseline hazard / censoring KM) — never the test
    fold's own median, which would leak test-set information into the group
    definition itself.

    Returns a boolean mask over `risks`; True = high-risk group.
    """
    threshold = np.median(np.asarray(train_risks, dtype=np.float64))
    return np.asarray(risks, dtype=np.float64) > threshold


# ── Smoke tests (validated against lifelines.statistics.logrank_test during
#    development — chi2 agreement to float precision) ────────────────────────
_rng = np.random.default_rng(11)
_n = 100

# Test 1: clearly different survival distributions -> expect small p-value
_t_hi = _rng.exponential(15, _n)   # shorter survival = high-risk group
_e_hi = _rng.binomial(1, 0.85, _n).astype(float)
_t_lo = _rng.exponential(40, _n)   # longer survival = low-risk group
_e_lo = _rng.binomial(1, 0.85, _n).astype(float)
_chi2, _p, _ome = logrank_test(_t_hi, _e_hi, _t_lo, _e_lo)
print(f"Test 1 (different groups)  : chi2={_chi2:.3f}  p={_p:.2e}  (expect p < 0.05)")
assert _p < 0.05, "Clearly different survival curves should be significant"

# Test 2: identical distributions -> expect a large p-value
_t_a = _rng.exponential(20, _n); _e_a = _rng.binomial(1, 0.8, _n).astype(float)
_t_b = _rng.exponential(20, _n); _e_b = _rng.binomial(1, 0.8, _n).astype(float)
_chi2_null, _p_null, _ = logrank_test(_t_a, _e_a, _t_b, _e_b)
print(f"Test 2 (same distribution) : chi2={_chi2_null:.3f}  p={_p_null:.2f}  (usually not significant)")

# Test 3: median-split helper produces two non-degenerate groups
_train_r = _rng.normal(0, 1, _n)
_test_r  = _rng.normal(0, 1, _n)
_hi_mask = median_split_groups(_train_r, _test_r)
print(f"Test 3 (median split)      : {_hi_mask.sum()} high-risk / {(~_hi_mask).sum()} low-risk")
assert 0 < _hi_mask.sum() < _n

print("\nSmoke tests passed.")
print("Log-rank test defined and smoke-tested.")


Test 1 (different groups)  : chi2=33.516  p=7.07e-09  (expect p < 0.05)
Test 2 (same distribution) : chi2=1.993  p=0.16  (usually not significant)
Test 3 (median split)      : 43 high-risk / 57 low-risk

Smoke tests passed.
Log-rank test defined and smoke-tested.


---
## Cell 4 — Data Loader Helper

Loads per-fold tensors from disk and returns PyTorch tensors on the target device.
Splits into train / val / test index sets.

In [19]:
# Cell 4 — Data Loader Helper (batched — exploits shared PPI topology)

def load_fold(fold_key, device):
    """Load all tensors for one fold. Returns dict of tensors."""
    fold_dir = DATA_SPLITS / fold_key

    feat   = torch.tensor(np.load(fold_dir / 'feature_tensor.npy'),  dtype=torch.float32)
    clin   = torch.tensor(np.load(fold_dir / 'clinical_tensor.npy'), dtype=torch.float32)
    surv   = torch.tensor(np.load(fold_dir / 'survival_labels.npy'), dtype=torch.float32)
    ei     = torch.tensor(np.load(fold_dir / 'edge_index.npy'),      dtype=torch.long)
    ew     = torch.tensor(np.load(fold_dir / 'edge_weight.npy'),     dtype=torch.float32)

    with open(fold_dir / 'split_indices.json') as f:
        split_idx = json.load(f)

    # .tolist() → pure Python list → np.array() — never touches the torch/numpy bridge
    times  = np.array(surv[:, 0].tolist(), dtype=np.float64)
    events = np.array(surv[:, 1].tolist(), dtype=np.float64)

    return {
        'feat'       : feat,
        'clin'       : clin,
        'times'      : times,
        'events'     : events,
        'edge_index' : ei.to(device),
        'edge_weight': ew.to(device),
        'train_idx'  : split_idx['train'],
        'val_idx'    : split_idx['val'],
        'test_idx'   : split_idx['test'],
    }


def build_batch_graph(indices, data, device, edge_cache):
    """Combine several patients into ONE block-diagonal graph.

    All patients in a fold share the same PPI topology (edge_index/edge_weight),
    so instead of calling GATConv once per patient, we stack their node
    features and replicate the topology len(indices) times with node-id
    offsets — GATConv then processes the whole batch in a single parallel
    call (verified numerically equivalent to the per-patient loop). The
    batched edge_index/edge_weight for a given batch size is cached, since
    it's identical every time that size is seen within a fold.
    """
    B = len(indices)
    n_nodes = data['feat'].shape[1]

    x        = data['feat'][indices].reshape(B * n_nodes, -1).to(device, non_blocking=True)
    clinical = data['clin'][indices].to(device, non_blocking=True)

    if B not in edge_cache:
        base_ei = data['edge_index']                       # (2, n_edges), already on device
        base_ew = data['edge_weight']                       # (n_edges,),   already on device
        offsets = (torch.arange(B, device=device) * n_nodes).view(B, 1, 1)
        ei_rep  = base_ei.unsqueeze(0).expand(B, -1, -1) + offsets   # (B, 2, n_edges)
        edge_cache[B] = (
            ei_rep.permute(1, 0, 2).reshape(2, -1),          # (2, B*n_edges)
            base_ew.repeat(B),                               # (B*n_edges,)
        )
    edge_index, edge_weight = edge_cache[B]

    batch_vec = torch.repeat_interleave(torch.arange(B, device=device), n_nodes)
    return x, edge_index, edge_weight, clinical, batch_vec


def get_batch_risks(model, data, indices, device, edge_cache, chunk_size=None):
    """Run model on a list of sample indices, return risks as numpy array.
    Processes `chunk_size` patients per forward call (default EVAL_BATCH_SIZE)
    instead of one patient per call. No gradients needed here, so this can
    safely use a larger chunk than the training MICRO_BATCH.
    """
    if chunk_size is None:
        chunk_size = EVAL_BATCH_SIZE
    model.eval()
    risks = []
    with torch.no_grad():
        for start in range(0, len(indices), chunk_size):
            chunk = indices[start:start + chunk_size]
            x, edge_index, edge_weight, clinical, batch_vec = build_batch_graph(
                chunk, data, device, edge_cache)
            with autocast(enabled=USE_AMP):
                r, _ = model(x, edge_index, edge_weight, clinical, batch=batch_vec)
            risks.extend(r.float().cpu().tolist())   # .float() → fp32 before leaving GPU
    return np.array(risks, dtype=np.float64)


print("Data loader helpers defined (batched).")


Data loader helpers defined (batched).


---
## Cell 5 — VRAM Dry-Run Memory Check

Runs one batched micro-batch (4 samples processed as a single block-diagonal
graph) on fold 0. Reports peak VRAM usage.
**If VRAM > 85% of total → reduce MICRO_BATCH in config.**

Safe to skip if you have already run this before.


In [20]:

if DEVICE.type != 'cuda':
    print("CPU device — skipping VRAM check.")
else:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    with open(DATA_SPLITS / 'cv_folds_repeated_5x3.json') as f:
        cv_splits = json.load(f)

    first_key   = list(cv_splits.keys())[0]
    data_check  = load_fold(first_key, DEVICE)
    model_check = GATv4().to(DEVICE)
    opt_check    = torch.optim.AdamW(model_check.parameters(), lr=LR_INITIAL)
    scaler_check = GradScaler(enabled=USE_AMP)
    edge_cache_check = {}

    model_check.train()
    opt_check.zero_grad()

    # Run ONE MICRO_BATCH-sized batched graph — matches exactly what the real
    # training loop now does per accumulation step (Cell 6).
    chunk = data_check['train_idx'][:MICRO_BATCH]
    x, edge_index, edge_weight, clinical, batch_vec = build_batch_graph(
        chunk, data_check, DEVICE, edge_cache_check)

    with autocast(enabled=USE_AMP):
        batch_risks = model_check(x, edge_index, edge_weight, clinical, batch=batch_vec)[0]
    batch_risks  = batch_risks.float()
    batch_times  = torch.tensor(data_check['times'][chunk],  device=DEVICE, dtype=torch.float32)
    batch_events = torch.tensor(data_check['events'][chunk], device=DEVICE, dtype=torch.float32)
    loss = cox_partial_likelihood_loss(batch_risks, batch_times, batch_events)

    scaler_check.scale(loss).backward()

    peak_mb = torch.cuda.max_memory_allocated() / 1024**2
    total_mb = torch.cuda.get_device_properties(0).total_memory / 1024**2

    print(f"Peak VRAM used  : {peak_mb:.0f} MB")
    print(f"Total VRAM      : {total_mb:.0f} MB")
    print(f"Headroom        : {total_mb - peak_mb:.0f} MB")

    del model_check, opt_check, scaler_check, data_check
    torch.cuda.empty_cache()

    if peak_mb > total_mb * 0.85:
        print(f"\nWARNING: Using >{0.85*100:.0f}% of VRAM on a single micro-batch.")
        print("  Reduce MICRO_BATCH from 4 to 2 in src/config.py before training.")
    else:
        print(f"\nVRAM check passed. Safe to run full 15-fold training.")


Peak VRAM used  : 145 MB
Total VRAM      : 4096 MB
Headroom        : 3950 MB

VRAM check passed. Safe to run full 15-fold training.


---
## Cell 6 — Training Loop (All 15 Folds)

**Resumable:** folds with an existing `best_model.pt` checkpoint are skipped.
Delete a checkpoint file to force retraining of that fold.

**Progress:** tqdm bar per fold + printed epoch-level loss/C-index.

**Batching:** each micro-batch of 4 patients is combined into one block-diagonal
graph (all patients share the same PPI topology) and run through GATConv in a
single parallel call, instead of 4 sequential single-patient calls. Combined
with mixed precision, this is a speed-only change — same effective batch size,
same accumulation schedule, same optimizer steps as before, so results should
match prior runs on this GPU up to normal floating-point run-to-run noise.
Actual wall-clock time depends on your GPU and PPI graph size — worth timing
one fold before assuming the full 15-fold runtime.


In [21]:
def make_lr_scheduler(optimizer, n_epochs, warmup_epochs, lr_init, lr_final):
    """Linear warmup + cosine decay scheduler."""
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return epoch / max(1, warmup_epochs)          # linear warmup
        progress = (epoch - warmup_epochs) / max(1, n_epochs - warmup_epochs)
        cosine   = 0.5 * (1 + math.cos(math.pi * progress))
        return lr_final / lr_init + (1 - lr_final / lr_init) * cosine
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


torch.manual_seed(MASTER_SEED)
np.random.seed(MASTER_SEED)

with open(DATA_SPLITS / 'cv_folds_repeated_5x3.json') as f:
    cv_splits = json.load(f)

all_fold_results = []
t_total_start    = time.time()

for fold_key, fold_def in cv_splits.items():
    ckpt_path = CKPT_DIR / f"{fold_key}_best_model.pt"
    result_path = CKPT_DIR / f"{fold_key}_result.json"

    # ── Resumability: skip if already done ───────────────────────────────────
    if result_path.exists():
        with open(result_path) as f:
            prev = json.load(f)
        print(f"[SKIP] {fold_key}  (C-index={prev['test_cindex']:.4f}, "
              f"IBS={prev.get('test_ibs', float('nan')):.4f}, "
              f"iAUC={prev.get('test_iauc', float('nan')):.4f}, "
              f"logrank_p={prev.get('test_logrank_p', float('nan')):.3g}, already done)")
        all_fold_results.append(prev)
        continue

    print(f"\n{'='*55}")
    print(f"Training fold: {fold_key}")
    print(f"{'='*55}")
    t_fold_start = time.time()

    # ── Load data ─────────────────────────────────────────────────────────────
    data = load_fold(fold_key, DEVICE)
    train_idx = data['train_idx']
    val_idx   = data['val_idx']
    test_idx  = data['test_idx']

    # Batched-edge-index cache — same PPI topology reused for every micro-batch
    # of a given size within this fold (built once per size, not rebuilt every
    # step).
    edge_cache = {}

    # ── Model + optimiser ─────────────────────────────────────────────────────
    # Seed per fold for reproducibility across seeds
    fold_seed = fold_def['seed'] + fold_def['fold'] * 100
    torch.manual_seed(fold_seed)

    model     = GATv4().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_INITIAL, weight_decay=1e-4)
    scheduler = make_lr_scheduler(optimizer, MAX_EPOCHS, WARMUP_EPOCHS, LR_INITIAL, LR_FINAL)
    scaler    = GradScaler(enabled=USE_AMP)

    best_val_cindex  = -1.0
    patience_counter = 0
    epoch_log        = []

    # ── Epoch loop ────────────────────────────────────────────────────────────
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        optimizer.zero_grad()

        # Shuffle training indices each epoch
        rng    = np.random.default_rng(fold_seed + epoch)
        idx_sh = rng.permutation(train_idx)

        train_loss    = 0.0
        n_accum_steps = 0

        # Gradient accumulation: process MICRO_BATCH samples AS ONE BATCHED
        # GRAPH (previously: MICRO_BATCH sequential single-patient forward
        # passes), accumulate ACCUM_STEPS times. Same effective batch
        # (MICRO_BATCH x ACCUM_STEPS) and same accumulation/step schedule as
        # before — only the forward-pass mechanics changed.
        idx_acc          = []
        batch_times_acc  = []
        batch_events_acc = []

        for step, i in enumerate(idx_sh):
            idx_acc.append(int(i))
            batch_times_acc.append(data['times'][i])
            batch_events_acc.append(data['events'][i])

            if len(idx_acc) == MICRO_BATCH:
                x, edge_index, edge_weight, clinical, batch_vec = build_batch_graph(
                    idx_acc, data, DEVICE, edge_cache)

                with autocast(enabled=USE_AMP):
                    risks_t = model(x, edge_index, edge_weight, clinical, batch=batch_vec)[0]
                risks_t  = risks_t.float()   # loss (log-cumsum-exp) computed in fp32 for stability
                times_t  = torch.tensor(batch_times_acc,  device=DEVICE, dtype=torch.float32)
                events_t = torch.tensor(batch_events_acc, device=DEVICE, dtype=torch.float32)

                loss = cox_partial_likelihood_loss(risks_t, times_t, events_t)
                loss = loss / ACCUM_STEPS   # scale for accumulation
                scaler.scale(loss).backward()

                train_loss += loss.item() * ACCUM_STEPS
                n_accum_steps += 1
                idx_acc          = []
                batch_times_acc  = []
                batch_events_acc = []

                if n_accum_steps % ACCUM_STEPS == 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()

        # Final gradient step if leftover samples
        if idx_acc:
            x, edge_index, edge_weight, clinical, batch_vec = build_batch_graph(
                idx_acc, data, DEVICE, edge_cache)
            with autocast(enabled=USE_AMP):
                risks_t = model(x, edge_index, edge_weight, clinical, batch=batch_vec)[0]
            risks_t  = risks_t.float()
            times_t  = torch.tensor(batch_times_acc,  device=DEVICE, dtype=torch.float32)
            events_t = torch.tensor(batch_events_acc, device=DEVICE, dtype=torch.float32)
            loss = cox_partial_likelihood_loss(risks_t, times_t, events_t)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        scheduler.step()

        # ── Validation C-index ────────────────────────────────────────────────
        val_risks  = get_batch_risks(model, data, val_idx, DEVICE, edge_cache)
        val_times  = data['times'][val_idx]
        val_events = data['events'][val_idx]
        val_cindex = concordance_index(val_risks, val_times, val_events)

        avg_loss = train_loss / max(1, n_accum_steps)
        cur_lr   = scheduler.get_last_lr()[0] * LR_INITIAL

        epoch_log.append({
            'epoch': epoch, 'train_loss': round(avg_loss, 4),
            'val_cindex': round(val_cindex, 4), 'lr': round(cur_lr, 7)
        })

        # Print every 10 epochs
        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{MAX_EPOCHS}  loss={avg_loss:.4f}  "
                  f"val_C={val_cindex:.4f}  lr={cur_lr:.2e}")

        # ── Best model checkpoint ─────────────────────────────────────────────
        if val_cindex > best_val_cindex:
            best_val_cindex  = val_cindex
            patience_counter = 0
            torch.save({
                'epoch'       : epoch,
                'model_state' : model.state_dict(),
                'val_cindex'  : val_cindex,
                'fold_key'    : fold_key,
            }, ckpt_path)
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  Early stop at epoch {epoch} (patience={PATIENCE})")
                break

    # ── Test evaluation on best checkpoint ────────────────────────────────────
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state'])

    test_risks  = get_batch_risks(model, data, test_idx, DEVICE, edge_cache)
    test_times  = data['times'][test_idx]
    test_events = data['events'][test_idx]
    test_cindex = concordance_index(test_risks, test_times, test_events)
    ci_lo, ci_hi = bootstrap_ci(test_risks, test_times, test_events)

    # ── Integrated Brier Score (IBS) ───────────────────────────────────────────
    # Baseline hazard + censoring KM are fit on this fold's TRAIN split only
    # (see Cell 3B), same as the model itself -- no test-fold leakage.
    train_risks = get_batch_risks(model, data, train_idx, DEVICE, edge_cache)
    fold_ibs, ibs_eval_times, ibs_curve = integrated_brier_score(
        train_risks, data['times'][train_idx], data['events'][train_idx],
        test_risks,  test_times,               test_events
    )
    np.savez(CKPT_DIR / f"{fold_key}_ibs_curve.npz",
             eval_times=ibs_eval_times, brier=ibs_curve)

    # ── Time-Dependent ROC-AUC (Uno's IPCW) ────────────────────────────────────
    # Same train-only-fit convention as IBS above (see Cell 3C).
    fold_iauc, tdauc_eval_times, tdauc_curve = integrated_time_dependent_auc(
        data['times'][train_idx], data['events'][train_idx],
        test_risks,               test_times,               test_events
    )
    np.savez(CKPT_DIR / f"{fold_key}_tdauc_curve.npz",
             eval_times=tdauc_eval_times, auc=tdauc_curve)

    # ── Log-Rank Test (median-split risk-group stratification) ─────────────────
    # Threshold fit on TRAIN risks only, applied to the TEST fold — same
    # leakage-safe convention as the IBS baseline hazard / censoring KM
    # and the time-dependent AUC censoring KM above (see Cell 3D).
    hi_risk_mask = median_split_groups(train_risks, test_risks)
    logrank_chi2, logrank_p, logrank_ome = logrank_test(
        test_times[hi_risk_mask],  test_events[hi_risk_mask],
        test_times[~hi_risk_mask], test_events[~hi_risk_mask]
    )

    fold_elapsed = time.time() - t_fold_start
    print(f"\n  [{fold_key}] Test C-index = {test_cindex:.4f}  "
          f"95% CI [{ci_lo:.4f}, {ci_hi:.4f}]  |  IBS = {fold_ibs:.4f}  |  "
          f"iAUC = {fold_iauc:.4f}  |  logrank p = {logrank_p:.3g}  "
          f"(best val epoch {ckpt['epoch']}, {fold_elapsed/60:.1f} min)")

    result = {
        'fold_key'      : fold_key,
        'seed'          : fold_def['seed'],
        'fold'          : fold_def['fold'],
        'best_val_epoch': int(ckpt['epoch']),
        'best_val_cindex': round(float(best_val_cindex), 4),
        'test_cindex'   : round(float(test_cindex), 4),
        'test_ci_lo'    : round(ci_lo, 4),
        'test_ci_hi'    : round(ci_hi, 4),
        'test_ibs'      : round(float(fold_ibs), 4),
        'test_iauc'         : round(float(fold_iauc), 4),
        'test_logrank_chi2' : round(float(logrank_chi2), 4),
        'test_logrank_p'    : round(float(logrank_p), 6),
        'test_logrank_significant': bool(logrank_p < 0.05),
        'n_hi_risk'     : int(hi_risk_mask.sum()),
        'n_lo_risk'     : int((~hi_risk_mask).sum()),
        'n_epochs_run'  : epoch,
        'elapsed_min'   : round(fold_elapsed / 60, 1),
    }
    with open(result_path, 'w') as f:
        json.dump(result, f, indent=2)
    with open(CKPT_DIR / f"{fold_key}_epoch_log.json", 'w') as f:
        json.dump(epoch_log, f)

    all_fold_results.append(result)

    # Free GPU memory before next fold
    del model, optimizer, scheduler, scaler, data, edge_cache
    torch.cuda.empty_cache()
    log.info(f"{fold_key}: test C-index={test_cindex:.4f}  IBS={fold_ibs:.4f}")

print(f"\nAll folds done in {(time.time()-t_total_start)/60:.0f} min")

[SKIP] seed42_fold0  (C-index=0.7995, IBS=0.2212, iAUC=0.7457, logrank_p=5.5e-05, already done)
[SKIP] seed42_fold1  (C-index=0.7597, IBS=0.1669, iAUC=0.7190, logrank_p=0.00144, already done)
[SKIP] seed42_fold2  (C-index=0.8047, IBS=0.1168, iAUC=0.8812, logrank_p=0, already done)
[SKIP] seed42_fold3  (C-index=0.6323, IBS=0.1564, iAUC=0.7568, logrank_p=0.00975, already done)
[SKIP] seed42_fold4  (C-index=0.7821, IBS=0.1761, iAUC=0.7901, logrank_p=0.00014, already done)
[SKIP] seed123_fold0  (C-index=0.7286, IBS=0.1614, iAUC=0.7554, logrank_p=0.00683, already done)
[SKIP] seed123_fold1  (C-index=0.7838, IBS=0.1460, iAUC=0.8205, logrank_p=3.8e-05, already done)
[SKIP] seed123_fold2  (C-index=0.7910, IBS=0.2165, iAUC=0.7478, logrank_p=0.000682, already done)
[SKIP] seed123_fold3  (C-index=0.7958, IBS=0.1702, iAUC=0.7979, logrank_p=1.3e-05, already done)
[SKIP] seed123_fold4  (C-index=0.8170, IBS=0.1091, iAUC=0.8815, logrank_p=2e-06, already done)
[SKIP] seed456_fold0  (C-index=0.7760, IBS

---
## Cell 7 — Aggregate Results

Computes the headline C-index **and IBS** reported in the paper:
- Mean ± std across 15 folds for both metrics
- Combined 95% CI via pooled bootstrap samples (C-index)

**Target:** mean C-index 0.72–0.76, std < 0.06

In [22]:
# Load all results (handles case where Cell 6 was resumed across sessions)
all_fold_results = []
for fold_key in cv_splits.keys():
    rp = CKPT_DIR / f"{fold_key}_result.json"
    if rp.exists():
        with open(rp) as f:
            all_fold_results.append(json.load(f))

results_df = pd.DataFrame(all_fold_results)
print(f"Folds with results: {len(results_df)} / {len(cv_splits)}")
print()

if len(results_df) == 0:
    print("No results found — Cell 6 may not have completed yet.")
else:
    print(results_df[[
        'fold_key','seed','fold','best_val_epoch',
        'best_val_cindex','test_cindex','test_ci_lo','test_ci_hi',
        'test_ibs','test_iauc','test_logrank_p','elapsed_min'
    ]].to_string(index=False))
    print()

    cindices = results_df['test_cindex'].values
    mean_ci  = cindices.mean()
    std_ci   = cindices.std()
    min_ci   = cindices.min()
    max_ci   = cindices.max()

    ibs_vals = results_df['test_ibs'].values
    mean_ibs = ibs_vals.mean()
    std_ibs  = ibs_vals.std()
    min_ibs  = ibs_vals.min()
    max_ibs  = ibs_vals.max()

    iauc_vals = results_df['test_iauc'].values
    mean_iauc = iauc_vals.mean()
    std_iauc  = iauc_vals.std()
    min_iauc  = iauc_vals.min()
    max_iauc  = iauc_vals.max()

    logrank_p_vals   = results_df['test_logrank_p'].values
    n_significant    = int(results_df['test_logrank_significant'].sum())
    frac_significant = n_significant / len(results_df)
    # Fisher's method: combine independent per-fold p-values into one
    # omnibus statistic for "does risk-group stratification generalize
    # across folds", rather than eyeballing 15 separate p-values.
    fisher_stat = -2.0 * np.sum(np.log(np.clip(logrank_p_vals, 1e-300, 1.0)))
    fisher_p    = float(1.0 - stats.chi2.cdf(fisher_stat, df=2 * len(logrank_p_vals)))

    print("=" * 50)
    print(f"  Mean C-index : {mean_ci:.4f}")
    print(f"  Std          : {std_ci:.4f}  (target < 0.06)")
    print(f"  Range        : [{min_ci:.4f}, {max_ci:.4f}]")
    print("-" * 50)
    print(f"  Mean IBS     : {mean_ibs:.4f}  (lower is better; 0.25 = uninformative)")
    print(f"  Std          : {std_ibs:.4f}")
    print(f"  Range        : [{min_ibs:.4f}, {max_ibs:.4f}]")
    print("-" * 50)
    print(f"  Mean iAUC    : {mean_iauc:.4f}  (time-dependent AUC, higher is better)")
    print(f"  Std          : {std_iauc:.4f}")
    print(f"  Range        : [{min_iauc:.4f}, {max_iauc:.4f}]")
    print("-" * 50)
    print(f"  Log-rank     : {n_significant}/{len(results_df)} folds significant at p<0.05 "
          f"({frac_significant:.0%})")
    print(f"  Fisher-combined p (across folds) : {fisher_p:.3g}")
    print("=" * 50)

    if std_ci > 0.06:
        print("WARNING: std > 0.06 — model may be unstable.")
        print("  Possible causes: small val set, class imbalance per fold.")
        print("  Consider: increase PATIENCE, check event rate per fold.")
    else:
        print("Std within acceptable range.")

    # Per-seed breakdown
    print(f"\nPer-seed mean C-index:")
    for seed, grp in results_df.groupby('seed'):
        print(f"  seed={seed}:  {grp['test_cindex'].mean():.4f} "
              f"(±{grp['test_cindex'].std():.4f})")

    print(f"\nPer-seed mean IBS:")
    for seed, grp in results_df.groupby('seed'):
        print(f"  seed={seed}:  {grp['test_ibs'].mean():.4f} "
              f"(±{grp['test_ibs'].std():.4f})")

    print(f"\nPer-seed mean iAUC:")
    for seed, grp in results_df.groupby('seed'):
        print(f"  seed={seed}:  {grp['test_iauc'].mean():.4f} "
              f"(±{grp['test_iauc'].std():.4f})")

    # Training efficiency
    print(f"\nTraining efficiency:")
    print(f"  Mean epochs run   : {results_df['n_epochs_run'].mean():.0f}")
    print(f"  Mean best epoch   : {results_df['best_val_epoch'].mean():.0f}")
    print(f"  Total wall time   : {results_df['elapsed_min'].sum():.0f} min")

    # Flag if early stopping never triggered (model may be undertrained)
    max_epochs_reached = (results_df['n_epochs_run'] >= MAX_EPOCHS).sum()
    if max_epochs_reached > 0:
        print(f"\n  NOTE: {max_epochs_reached} folds hit max_epochs without early stop.")
        print("    Consider increasing MAX_EPOCHS in config.py.")

Folds with results: 15 / 15

     fold_key  seed  fold  best_val_epoch  best_val_cindex  test_cindex  test_ci_lo  test_ci_hi  test_ibs  test_iauc  test_logrank_p  elapsed_min
 seed42_fold0    42     0              32           0.7593       0.7995      0.7090      0.8667    0.2212     0.7457        0.000055          4.6
 seed42_fold1    42     1              29           0.7993       0.7597      0.6710      0.8349    0.1669     0.7190        0.001444          3.9
 seed42_fold2    42     2              28           0.7625       0.8047      0.7302      0.8805    0.1168     0.8812        0.000000          4.1
 seed42_fold3    42     3              28           0.7950       0.6323      0.5431      0.7301    0.1564     0.7568        0.009752          3.9
 seed42_fold4    42     4              41           0.7710       0.7821      0.7091      0.8522    0.1761     0.7901        0.000140          4.7
seed123_fold0   123     0              33           0.8295       0.7286      0.6460      0.8060

---
## Cell 8 — Save Results + Audit YAML

In [23]:
# Save results table
results_path = TABLE_DIR / 'cv_results.csv'
results_df.to_csv(results_path, index=False)
print(f"Results table -> {results_path}")

audit = {
    'phase'    : 'Phase 7 — Model Training (15-fold CV)',
    'timestamp': datetime.now().isoformat(),

    'model': {
        'architecture' : 'GATv4 — 2-layer GAT + residual + MLP risk head',
        'in_dim'       : GAT_IN_DIM,
        'hidden'       : GAT_HIDDEN,
        'heads'        : GAT_HEADS,
        'mlp_hidden'   : GAT_MLP_HIDDEN,
        'dropout'      : GAT_DROPOUT,
        'n_parameters' : int(sum(p.numel() for p in GATv4().parameters())),
    },

    'training': {
        'loss'          : 'Cox partial likelihood (Breslow approximation)',
        'optimizer'     : 'AdamW (lr 1e-3 → 1e-5, cosine, 10-epoch warmup)',
        'effective_batch': MICRO_BATCH * ACCUM_STEPS,
        'max_epochs'    : MAX_EPOCHS,
        'patience'      : PATIENCE,
        'grad_clip'     : 1.0,
        'device'        : str(DEVICE),
    },

    'results': {
        'n_folds_complete' : len(results_df),
        'mean_test_cindex' : round(float(mean_ci), 4),
        'std_test_cindex'  : round(float(std_ci),  4),
        'min_test_cindex'  : round(float(min_ci),  4),
        'max_test_cindex'  : round(float(max_ci),  4),
        'target_range'     : '0.72–0.76',
        'stable'           : bool(std_ci <= 0.06),
        'mean_test_ibs'    : round(float(mean_ibs), 4),
        'std_test_ibs'     : round(float(std_ibs),  4),
        'min_test_ibs'     : round(float(min_ibs),  4),
        'max_test_ibs'     : round(float(max_ibs),  4),
        'mean_test_iauc'   : round(float(mean_iauc), 4),
        'std_test_iauc'    : round(float(std_iauc),  4),
        'min_test_iauc'    : round(float(min_iauc),  4),
        'max_test_iauc'    : round(float(max_iauc),  4),
        'logrank_folds_significant' : f"{n_significant}/{len(results_df)}",
        'logrank_fisher_combined_p' : round(float(fisher_p), 6),
        'mean_best_epoch'  : round(float(results_df['best_val_epoch'].mean()), 1),
        'total_train_min'  : round(float(results_df['elapsed_min'].sum()), 1),
    },

    'per_fold': results_df.to_dict(orient='records'),

    'next_phase' : 'Phase 8 — Pathway Analysis + Biomarker Discovery',
    'status'     : 'COMPLETE',
}

audit_path = AUDIT_DIR / 'phase7_training.yaml'
with open(audit_path, 'w', encoding='utf-8') as f:
    yaml.dump(audit, f, default_flow_style=False, sort_keys=False)
print(f"Audit YAML -> {audit_path}")

print()
print("=" * 55)
print("PHASE 7 COMPLETE")
print("=" * 55)
print(f"  Folds trained  : {len(results_df)} / {len(cv_splits)}")
print(f"  Mean C-index   : {mean_ci:.4f} ± {std_ci:.4f}")
print(f"  Range          : [{min_ci:.4f}, {max_ci:.4f}]")
print(f"  Mean IBS       : {mean_ibs:.4f} ± {std_ibs:.4f}")
print(f"  Range          : [{min_ibs:.4f}, {max_ibs:.4f}]")
print(f"  Mean iAUC      : {mean_iauc:.4f} ± {std_iauc:.4f}")
print(f"  Range          : [{min_iauc:.4f}, {max_iauc:.4f}]")
print(f"  Log-rank sig.  : {n_significant}/{len(results_df)} folds (p<0.05)  |  "
      f"Fisher-combined p = {fisher_p:.3g}")
print()
print("Next: pathway analysis + integrated gradients biomarker discovery")

Results table -> C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\results\tables\cv_results.csv
Audit YAML -> C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\audit\phase7_training.yaml

PHASE 7 COMPLETE
  Folds trained  : 15 / 15
  Mean C-index   : 0.7697 ± 0.0442
  Range          : [0.6323, 0.8170]
  Mean IBS       : 0.1615 ± 0.0320
  Range          : [0.1091, 0.2212]
  Mean iAUC      : 0.7913 ± 0.0477
  Range          : [0.7190, 0.8815]
  Log-rank sig.  : 15/15 folds (p<0.05)  |  Fisher-combined p = 0

Next: pathway analysis + integrated gradients biomarker discovery
